# agent_framework_azure_ai_search

## Example 1: a semantic-search context provider

In [ ]:
# Example 1: the provider keeps the endpoint, index and search mode it was given
from agent_framework_azure_ai_search import AzureAISearchContextProvider

provider = AzureAISearchContextProvider(endpoint="https://example.search.windows.net", index_name="docs", api_key="test-key", top_k=3)
assert provider.index_name == "docs" and provider.mode == "semantic"
assert provider.endpoint == "https://example.search.windows.net"
print("Example 1:", type(provider).__name__, provider.index_name, provider.mode)

## Example 2: settings from AZURE_SEARCH_* environment variables

In [ ]:
# Example 2: with no endpoint argument, the provider reads endpoint, index and key from the environment
import os
from unittest import mock
from agent_framework_azure_ai_search import AzureAISearchContextProvider

env = {"AZURE_SEARCH_ENDPOINT": "https://from-env.search.windows.net", "AZURE_SEARCH_INDEX_NAME": "env-index", "AZURE_SEARCH_API_KEY": "k"}
with mock.patch.dict(os.environ, env):
    provider = AzureAISearchContextProvider()
assert provider.index_name == "env-index", provider.index_name
print("Example 2: index from the environment:", provider.index_name)

## Example 3: a custom context prompt

In [ ]:
# Example 3: the text placed before retrieved documents can be replaced
from agent_framework_azure_ai_search import AzureAISearchContextProvider

prompt = "Use only these passages:"
provider = AzureAISearchContextProvider(endpoint="https://example.search.windows.net", index_name="docs", api_key="k", context_prompt=prompt)
assert provider.context_prompt == prompt
print("Example 3: context prompt:", provider.context_prompt)

## Example 4: the provider attaches to an Agent

In [ ]:
# Example 4: an Agent takes the provider as a context provider; retrieval runs only when the agent does
from agent_framework import Agent, ChatResponse, Message
from agent_framework_azure_ai_search import AzureAISearchContextProvider

class NoModel:
    additional_properties: dict = {}
    async def get_response(self, messages, **kwargs):
        return ChatResponse(messages=Message(role="assistant", contents=["unused"]))

provider = AzureAISearchContextProvider(endpoint="https://example.search.windows.net", index_name="docs", api_key="k")
agent = Agent(client=NoModel(), name="rag_bot", context_providers=[provider])
assert agent.name == "rag_bot"
print("Example 4: agent", agent.name, "with", type(provider).__name__)

## Example 5: `agent_framework.azure` re-exports this package

In [ ]:
# Example 5: the umbrella namespace for Azure connectors hands back the same class
import importlib.metadata
import agent_framework.azure as via_core
import agent_framework_azure_ai_search as direct

assert via_core.AzureAISearchContextProvider is direct.AzureAISearchContextProvider
print("Example 5:", importlib.metadata.version("agent-framework-azure-ai-search"), "re-exported through agent_framework.azure")